# Heart Disease Prediction — EDA & Model Training

This notebook explores the Cleveland Heart Disease dataset and trains classifiers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load Data

In [ ]:
df = pd.read_csv('Data/heart.csv')
print(df.shape)
df.head()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print()
print('Target distribution:')
print(df['target'].value_counts())

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['target'].value_counts().plot.pie(ax=axes[0], autopct='%1.1f%%', labels=['No Disease', 'Disease'],
                                      colors=['#27ae60','#e74c3c'], startangle=90)
axes[0].set_title('Target Distribution')
axes[0].set_ylabel('')
sns.histplot(data=df, x='age', hue='target', kde=True, ax=axes[1], palette=['#27ae60','#e74c3c'])
axes[1].set_title('Age Distribution by Target')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
num_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(1, len(num_features), figsize=(18, 4))
for ax, col in zip(axes, num_features):
    sns.boxplot(data=df, x='target', y=col, ax=ax,
                palette={0: '#27ae60', 1: '#e74c3c'})
    ax.set_title(col)
    ax.set_xlabel('Target (0=No Disease, 1=Disease)')
plt.suptitle('Numerical Features by Target', y=1.02)
plt.tight_layout()
plt.show()

## 3. Model Training & Comparison

In [ ]:
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'SVM': SVC(kernel='rbf', C=2, probability=True, random_state=42),
}

results = []
for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', model)])
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='f1')
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results.append({
        'Model': name,
        'CV F1 (mean)': round(cv_scores.mean(), 4),
        'CV F1 (std)': round(cv_scores.std(), 4),
        'Test Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, pipe.predict_proba(X_test)[:,1]), 4),
    })

results_df = pd.DataFrame(results).sort_values('CV F1 (mean)', ascending=False)
results_df

In [ ]:
# ROC curves
plt.figure(figsize=(8, 6))
for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', model)])
    pipe.fit(X_train, y_train)
    fpr, tpr, _ = roc_curve(y_test, pipe.predict_proba(X_test)[:,1])
    auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:,1])
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.2f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

## 4. Best Model — Feature Importance

In [ ]:
best_pipe = Pipeline([('scaler', StandardScaler()),
                      ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])
best_pipe.fit(X_train, y_train)
importances = best_pipe['clf'].feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
feat_imp.plot(kind='bar', color='#e74c3c', edgecolor='white')
plt.title('Feature Importances — Random Forest')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()